In [1]:
"""
Week 10 - Function 1 (TRANSPARENCY + INTERPRETABILITY-AWARE)
Upgrades vs Week 9:
- Full "decision trace": prints top EI candidates (x, EI, PI, pred mean/std), shortlist stats, coverage stats
- Coverage audit (nearest-neighbour distance + spread) to reveal sampling bias / gaps
- Adaptive exploration mix: slightly increases global exploration if dataset is clustered
- Interpretable local sensitivity: gradient of posterior-mean predictive mean wrt x at chosen point
- Strict [0,1]^2 clamp + 6-decimal output policy (dedup uses same rounding)
"""

import math
import numpy as np
from dataclasses import dataclass

import torch
import torch.nn as nn

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold

import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO, Predictive
from pyro.infer.autoguide import AutoDiagonalNormal
from pyro.optim import Adam


# ------------------------ 1. Reproducibility + Base Config ------------------------

RANDOM_SEED = 123
INPUT_DIM = 2
HIDDEN_SIZES = [64, 64]

# Feature extractor training
N_EPOCHS_FEATURE = 1400
LR_FEATURE = 1e-3

# Hyperparam tuning (kept modest for compute)
N_TUNING_TRIALS = 18
N_FOLDS = 4

# Output / dedup precision
DUP_DECIMALS = 6

# Candidate budgets scale with n (diminishing returns control)
def scaled_budget(n: int) -> int:
    # ~ 28k at n=18; ~ 45k at n=40; hard cap 60k
    return int(min(60000, max(22000, 1500 * n + 1000)))

# Trust-region parameters (local maxima focus)
N_BEST_ANCHORS = 3
LOCAL_SIGMA = 0.035  # exploit radius
LOCAL_MIX_BASE = 0.70  # baseline % candidates from local trust region

# Two-stage acquisition (scaling-aware)
N_PRED_SAMPLES_CHEAP = 128
N_PRED_SAMPLES_EXPENSIVE = 640
SHORTLIST_M = 256

# Decoding / selection temperature (flat-EI guard)
TOP_K = 32
TOP_P = 0.80
TEMP_GREEDY = 0.05
TEMP_FLAT = 0.35
EI_FLAT_RATIO = 1.08
SAFE_GLOBAL_JUMP_SIGMA = 0.18

# Fallback defaults
DEFAULT_PRIOR_SCALE = 1.0
DEFAULT_LR_VI = 3e-3
DEFAULT_N_STEPS_VI = 4500
DEFAULT_N_PRED_SAMPLES = 256
DEFAULT_XI = 0.0025

# Week 10 interpretability printing
TOP_REPORT_K = 5  # show top-k candidates by EI from expensive stage


def set_all_seeds(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)
    pyro.set_rng_seed(seed)


# ------------------------ 2. Data ------------------------

def load_data():
    X_raw = np.array([
        [0.31940389, 0.76295937],
        [0.57432921, 0.87989810],
        [0.73102363, 0.73299988],
        [0.84035342, 0.26473161],
        [0.65011406, 0.68152635],
        [0.41043714, 0.14755430],
        [0.31269116, 0.07872278],
        [0.68341817, 0.86105746],
        [0.08250725, 0.40348751],
        [0.88388983, 0.58225397],
        [0.88389,    0.98389],
        [0.37454,    0.950713],
        [0.382224,   0.951319],
        [0.782778,   0.793329],
        [0.030500,   0.037300],
        [0.646168,   0.172681],
        [0.546683,   0.562315],
        [0.814946,   0.846025],
        [0.480813,   0.998897]
    ], dtype=np.float64)

    y_raw = np.array([
        1.32267704e-79,
        1.03307824e-46,
        7.71087511e-16,
        3.34177101e-124,
        -3.60606264e-03,
        -2.15924904e-54,
        -2.08909327e-91,
        2.53500115e-40,
        3.60677119e-81,
        6.22985647e-48,
        9.59033053e-135,
        -1.56227724e-117,
        -4.77166224e-115,
        7.535209723645751e-36,
        1.6357533426693436e-209,
        6.327028545366271e-79,
        3.45699404834516e-08,
        -5.636591868482337e-58,
        -3.898697523271252e-111
    ], dtype=np.float64)

    assert X_raw.shape[1] == INPUT_DIM
    return X_raw, y_raw


# ------------------------ 3. Signed-log transform for y ------------------------

def signed_log_transform(y: np.ndarray, s: float):
    return np.sign(y) * np.log1p(np.abs(y) / s)

def signed_log_inverse(y_t: np.ndarray, s: float):
    return np.sign(y_t) * s * np.expm1(np.abs(y_t))

def compute_scale_s(y: np.ndarray):
    med = float(np.median(np.abs(y)))
    return max(med, 1e-12)


# ------------------------ 4. Feature extractor (deterministic) ------------------------

class FeatureExtractor(nn.Module):
    def __init__(self, input_dim: int, hidden_sizes):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_sizes:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.ReLU())
            prev_dim = h
        self.net = nn.Sequential(*layers)
        self.output_dim = prev_dim

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

class DeterministicRegressor(nn.Module):
    def __init__(self, input_dim: int, hidden_sizes):
        super().__init__()
        self.feature_extractor = FeatureExtractor(input_dim, hidden_sizes)
        self.head = nn.Linear(self.feature_extractor.output_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feats = self.feature_extractor(x)
        return self.head(feats).squeeze(-1)

def train_feature_extractor(X: torch.Tensor, y: torch.Tensor) -> FeatureExtractor:
    set_all_seeds(RANDOM_SEED)
    model = DeterministicRegressor(INPUT_DIM, HIDDEN_SIZES)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR_FEATURE)
    loss_fn = nn.MSELoss()

    model.train()
    for _ in range(N_EPOCHS_FEATURE):
        optimizer.zero_grad()
        preds = model(X)
        loss = loss_fn(preds, y)
        loss.backward()
        optimizer.step()

    feature_extractor = FeatureExtractor(INPUT_DIM, HIDDEN_SIZES)
    feature_extractor.load_state_dict(model.feature_extractor.state_dict())
    return feature_extractor


# ------------------------ 5. Bayesian linear head (Pyro) ------------------------

@dataclass
class BayesianHeadConfig:
    prior_scale: float

def make_bayesian_model(feature_extractor: FeatureExtractor, cfg: BayesianHeadConfig):
    def model(x, y=None):
        pyro.module("feature_extractor", feature_extractor, update_module_params=False)
        feats = feature_extractor(x)
        H = feats.size(-1)

        weight = pyro.sample(
            "weight",
            dist.Normal(x.new_zeros(H), cfg.prior_scale * x.new_ones(H)).to_event(1)
        )
        bias = pyro.sample("bias", dist.Normal(x.new_tensor(0.0), cfg.prior_scale))
        sigma = pyro.sample("sigma", dist.HalfCauchy(x.new_tensor(1.0)))

        mean = (feats * weight).sum(dim=-1) + bias
        with pyro.plate("data", x.size(0)):
            pyro.sample("obs", dist.Normal(mean, sigma), obs=y)

    return model

def train_bayesian_head(model, X: torch.Tensor, y: torch.Tensor, lr_vi: float, n_steps_vi: int):
    pyro.clear_param_store()
    guide = AutoDiagonalNormal(model)
    optimizer = Adam({"lr": lr_vi})
    svi = SVI(model, guide, optimizer, loss=Trace_ELBO())
    for _ in range(n_steps_vi):
        svi.step(X, y)
    return guide


# ------------------------ 6. Acquisition helpers ------------------------

def normal_cdf(x: torch.Tensor) -> torch.Tensor:
    return 0.5 * (1.0 + torch.erf(x / math.sqrt(2.0)))

def normal_pdf(x: torch.Tensor) -> torch.Tensor:
    return (1.0 / math.sqrt(2.0 * math.pi)) * torch.exp(-0.5 * x**2)

def compute_ei_and_pi(samples: torch.Tensor, best_y: float, xi: float):
    mean = samples.mean(dim=0)
    std = samples.std(dim=0) + 1e-9
    gamma = (mean - best_y - xi) / std
    ei = (mean - best_y - xi) * normal_cdf(gamma) + std * normal_pdf(gamma)
    ei = torch.clamp(ei, min=0.0)
    pi = normal_cdf((mean - best_y - xi) / std)
    return ei, pi, mean, std


# ------------------------ 7. Tuning (Random Search + CV) ------------------------

def mc_predictive_mean_std(model, guide, X: torch.Tensor, n_samples: int):
    predictive = Predictive(model, guide=guide, num_samples=n_samples, return_sites=("obs",))
    with torch.no_grad():
        pred = predictive(X)["obs"]
    return pred.mean(dim=0), pred.std(dim=0) + 1e-9

def cv_score(feature_extractor: FeatureExtractor,
             X_all: torch.Tensor,
             y_all: torch.Tensor,
             prior_scale: float,
             lr_vi: float,
             n_steps_vi: int,
             n_pred_samples: int):
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    scores = []
    for train_idx, val_idx in kf.split(X_all.cpu().numpy()):
        X_tr, y_tr = X_all[train_idx], y_all[train_idx]
        X_va, y_va = X_all[val_idx], y_all[val_idx]

        cfg = BayesianHeadConfig(prior_scale=prior_scale)
        model = make_bayesian_model(feature_extractor, cfg)
        guide = train_bayesian_head(model, X_tr, y_tr, lr_vi=lr_vi, n_steps_vi=n_steps_vi)

        mu, sd = mc_predictive_mean_std(model, guide, X_va, n_samples=n_pred_samples)
        ll = (-0.5 * torch.log(2 * math.pi * sd**2) - 0.5 * ((y_va - mu) ** 2) / (sd**2)).mean()
        scores.append(ll.item())
    return float(np.mean(scores))

def random_log_uniform(rng, low, high):
    return float(np.exp(rng.uniform(np.log(low), np.log(high))))

def tune_hyperparameters(feature_extractor: FeatureExtractor, X_all: torch.Tensor, y_all: torch.Tensor):
    rng = np.random.default_rng(RANDOM_SEED)
    steps_choices = [2500, 3500, 4500, 6000]
    pred_choices = [256, 384, 512]

    best = {
        "score": -1e18,
        "prior_scale": DEFAULT_PRIOR_SCALE,
        "lr_vi": DEFAULT_LR_VI,
        "n_steps_vi": DEFAULT_N_STEPS_VI,
        "n_pred_samples": DEFAULT_N_PRED_SAMPLES,
        "xi": DEFAULT_XI,
    }

    for _ in range(N_TUNING_TRIALS):
        prior_scale = random_log_uniform(rng, 0.08, 4.0)
        lr_vi = random_log_uniform(rng, 8e-4, 8e-3)
        n_steps_vi = int(rng.choice(steps_choices))
        n_pred_samples = int(rng.choice(pred_choices))
        xi = random_log_uniform(rng, 5e-5, 8e-3)

        try:
            score = cv_score(feature_extractor, X_all, y_all,
                             prior_scale=prior_scale,
                             lr_vi=lr_vi,
                             n_steps_vi=n_steps_vi,
                             n_pred_samples=n_pred_samples)
        except Exception:
            continue

        if score > best["score"]:
            best.update({
                "score": score,
                "prior_scale": prior_scale,
                "lr_vi": lr_vi,
                "n_steps_vi": n_steps_vi,
                "n_pred_samples": n_pred_samples,
                "xi": xi,
            })
    return best


# ------------------------ 8. Week 10 transparency helpers ------------------------

def _rounded_key(x_np: np.ndarray, decimals: int = DUP_DECIMALS):
    return tuple(np.round(x_np.astype(np.float64), decimals=decimals).tolist())

def coverage_audit(X_raw: np.ndarray):
    """
    Interpretable space-coverage diagnostics:
    - nearest neighbour distance distribution (cluster vs spread)
    - coordinate-wise spread
    """
    X = X_raw.astype(np.float64)
    n = X.shape[0]
    # pairwise distances (O(n^2) is fine for n~20)
    D = np.sqrt(((X[:, None, :] - X[None, :, :]) ** 2).sum(axis=-1))
    np.fill_diagonal(D, np.inf)
    nn = D.min(axis=1)
    audit = {
        "nn_min": float(nn.min()),
        "nn_median": float(np.median(nn)),
        "nn_mean": float(nn.mean()),
        "spread_dim0": float(X[:, 0].max() - X[:, 0].min()),
        "spread_dim1": float(X[:, 1].max() - X[:, 1].min()),
    }
    return audit

def adaptive_local_mix_from_coverage(audit: dict) -> float:
    """
    If points are very clustered (small nn_median), increase global exploration a bit.
    This is transparent + helps reduce sampling bias.
    """
    mix = LOCAL_MIX_BASE
    # heuristics tuned for [0,1]^2
    if audit["nn_median"] < 0.06:
        mix = max(0.55, mix - 0.10)
    if audit["nn_median"] < 0.045:
        mix = max(0.50, mix - 0.10)
    return float(mix)

def build_candidates_week10(X_train_raw: np.ndarray, y_train_raw: np.ndarray, device: torch.device):
    """
    Trust-region-first with a transparent coverage audit and slight mix adaptation.
    """
    torch.manual_seed(RANDOM_SEED)
    n = len(X_train_raw)
    max_tokens = scaled_budget(n)

    audit = coverage_audit(X_train_raw)
    local_mix = adaptive_local_mix_from_coverage(audit)

    n_local = int(max_tokens * local_mix)
    n_global = max_tokens - n_local

    sob = torch.quasirandom.SobolEngine(dimension=INPUT_DIM, scramble=True, seed=RANDOM_SEED)
    Xg = sob.draw(n_global).to(device)

    best_idx = np.argsort(-y_train_raw)[:min(N_BEST_ANCHORS, n)]
    anchors = torch.from_numpy(X_train_raw[best_idx].astype(np.float32)).to(device)

    idx = torch.randint(low=0, high=anchors.size(0), size=(n_local,), device=device)
    noise = LOCAL_SIGMA * torch.randn((n_local, INPUT_DIM), device=device)
    Xl = torch.clamp(anchors[idx] + noise, 0.0, 1.0)

    Xcand = torch.cat([Xl, Xg], dim=0)

    train_keys = set(_rounded_key(x) for x in X_train_raw)
    seen = set(train_keys)
    keep = []

    Xcand_np = Xcand.detach().cpu().numpy()
    for i in range(Xcand_np.shape[0]):
        k = _rounded_key(Xcand_np[i])
        if k in seen:
            continue
        seen.add(k)
        keep.append(i)

    if len(keep) == 0:
        Xcand = torch.rand((min(4096, max_tokens), INPUT_DIM), device=device)
    else:
        Xcand = Xcand[torch.tensor(keep, device=device)]

    meta = {
        "max_tokens": max_tokens,
        "n_global": n_global,
        "n_local": n_local,
        "local_mix": local_mix,
        "coverage_audit": audit,
    }
    return Xcand, meta

def adaptive_temperature_from_ei(ei: torch.Tensor) -> float:
    if ei.numel() < 10:
        return TEMP_GREEDY
    top10 = torch.topk(ei, k=10).values
    denom = float(top10.mean().item()) + 1e-12
    ratio = float(top10[0].item()) / denom
    return TEMP_FLAT if ratio < EI_FLAT_RATIO else TEMP_GREEDY

def decode_pick_from_top(ei: torch.Tensor, temperature: float, top_k: int, top_p: float):
    ei = torch.clamp(ei, min=0.0)
    if ei.numel() == 0:
        return None

    k = min(int(top_k), int(ei.numel()))
    top_vals, top_idx = torch.topk(ei, k=k)

    if float(top_vals.sum().item()) <= 1e-12:
        return int(top_idx[0].item())

    probs = top_vals / top_vals.sum()
    sorted_probs, order = torch.sort(probs, descending=True)
    cum = torch.cumsum(sorted_probs, dim=0)
    m = int(torch.searchsorted(cum, torch.tensor(top_p, device=ei.device)).item()) + 1
    m = max(1, min(m, sorted_probs.numel()))

    nucleus_order = order[:m]
    nucleus_probs = sorted_probs[:m]

    T = max(1e-6, float(temperature))
    logits = torch.log(nucleus_probs + 1e-12) / T
    weights = torch.softmax(logits, dim=0)

    j = torch.multinomial(weights, num_samples=1).item()
    chosen_top_index = int(nucleus_order[j].item())
    chosen_global_index = int(top_idx[chosen_top_index].item())
    return chosen_global_index

def safe_global_jump(X_train_raw_np: np.ndarray, y_train_raw_np: np.ndarray):
    best_idx = int(np.argmax(y_train_raw_np))
    x_best = X_train_raw_np[best_idx]
    rng = np.random.default_rng(RANDOM_SEED + 999)
    x = x_best + SAFE_GLOBAL_JUMP_SIGMA * rng.normal(size=(INPUT_DIM,))
    x = np.clip(x, 0.0, 1.0)
    return x.astype(np.float64)


def posterior_mean_sensitivity(feature_extractor: FeatureExtractor, guide, x_raw_2d: np.ndarray, device: torch.device):
    """
    Interpretability: gradient of posterior-mean predictive mean wrt x at x_next.
    Uses guide.median() (a stable point estimate) for weight/bias.
    """
    med = guide.median()
    w = med["weight"].detach()  # shape [H]
    b = med["bias"].detach()    # scalar

    x = torch.tensor(x_raw_2d.reshape(1, -1), dtype=torch.float32, device=device, requires_grad=True)
    feats = feature_extractor(x)
    mu = (feats * w).sum(dim=-1) + b
    mu.backward()

    grad = x.grad.detach().cpu().numpy().reshape(-1)
    return {
        "grad_dim0": float(grad[0]),
        "grad_dim1": float(grad[1]),
        "grad_l2": float(np.sqrt((grad ** 2).sum())),
        "grad_abs_sum": float(np.abs(grad).sum())
    }


def propose_next_point_week10(model, guide,
                              X_train_scaled: torch.Tensor,
                              y_train_scaled: torch.Tensor,
                              X_train_raw_np: np.ndarray,
                              y_train_raw_np: np.ndarray,
                              scaler_y_transformed: StandardScaler,
                              y_scale_s: float,
                              xi: float):
    device = X_train_scaled.device

    # --- Candidate build + transparent meta ---
    X_candidates, meta = build_candidates_week10(X_train_raw_np, y_train_raw_np, device=device)

    # --- Stage 1: cheap EI screen ---
    predictive_cheap = Predictive(model, guide=guide, num_samples=N_PRED_SAMPLES_CHEAP, return_sites=("obs",))
    with torch.no_grad():
        samples_scaled_cheap = predictive_cheap(X_candidates)["obs"]

    best_y_scaled = float(y_train_scaled.max().item())
    ei_cheap, pi_cheap, mean_scaled_cheap, std_scaled_cheap = compute_ei_and_pi(
        samples_scaled_cheap, best_y_scaled, xi=xi
    )

    if ei_cheap.numel() == 0:
        x_fallback = safe_global_jump(X_train_raw_np, y_train_raw_np)
        return x_fallback, {"fallback": True, "meta": meta}

    m = min(int(SHORTLIST_M), int(ei_cheap.numel()))
    _, short_idx = torch.topk(ei_cheap, k=m)
    X_short = X_candidates[short_idx]

    # --- Stage 2: expensive EI re-rank ---
    predictive_exp = Predictive(model, guide=guide, num_samples=N_PRED_SAMPLES_EXPENSIVE, return_sites=("obs",))
    with torch.no_grad():
        samples_scaled_exp = predictive_exp(X_short)["obs"]

    ei, pi, mean_scaled, std_scaled = compute_ei_and_pi(samples_scaled_exp, best_y_scaled, xi=xi)

    temp = adaptive_temperature_from_ei(ei)
    idx_decoded = decode_pick_from_top(ei, temperature=temp, top_k=TOP_K, top_p=TOP_P)
    idx_best = int(torch.argmax(ei).item()) if idx_decoded is None else int(idx_decoded)

    next_x = X_short[idx_best].detach().cpu().numpy()
    next_x = np.clip(next_x.astype(np.float64), 0.0, 1.0)

    # flatness diagnostics
    top1 = float(torch.max(ei).item())
    top10_mean = float(torch.topk(ei, k=min(10, ei.numel())).values.mean().item())
    flat_ratio = (top1 / (top10_mean + 1e-12)) if ei.numel() >= 10 else 999.0

    if (ei.numel() >= 10) and (flat_ratio < EI_FLAT_RATIO) and (top1 < 1e-6):
        next_x = safe_global_jump(X_train_raw_np, y_train_raw_np)

    # --- Transform predictive stats (expensive stage) back to raw scale ---
    mean_scaled_np = mean_scaled.detach().cpu().numpy()
    std_scaled_np = std_scaled.detach().cpu().numpy()

    mean_trans = scaler_y_transformed.inverse_transform(mean_scaled_np.reshape(-1, 1)).squeeze(-1)
    mean_raw = signed_log_inverse(mean_trans, y_scale_s)

    std_trans = std_scaled_np * scaler_y_transformed.scale_[0]
    std_raw = np.abs(
        signed_log_inverse(mean_trans + std_trans, y_scale_s) - signed_log_inverse(mean_trans, y_scale_s)
    )

    best_y_trans = float(scaler_y_transformed.inverse_transform(np.array(best_y_scaled).reshape(-1, 1)).squeeze())
    best_y_raw = float(signed_log_inverse(np.array([best_y_trans]), y_scale_s).squeeze())

    # --- Build interpretable report (top contenders) ---
    short_np = X_short.detach().cpu().numpy().astype(np.float64)
    ei_np = ei.detach().cpu().numpy().astype(np.float64)
    pi_np = pi.detach().cpu().numpy().astype(np.float64)

    topk = min(TOP_REPORT_K, ei_np.shape[0])
    top_idx = np.argsort(-ei_np)[:topk]

    report_rows = []
    for j in top_idx:
        xj = np.clip(short_np[j], 0.0, 1.0)
        report_rows.append({
            "x": np.round(xj, DUP_DECIMALS).tolist(),
            "EI": float(ei_np[j]),
            "PI": float(pi_np[j]),
            "pred_mean_raw": float(mean_raw[j]),
            "pred_std_raw": float(std_raw[j]),
        })

    info = {
        "next_x": next_x,
        "best_y_raw": best_y_raw,
        "meta": meta,
        "decode_settings": {
            "temperature_used": temp,
            "top_p": TOP_P,
            "top_k": TOP_K,
            "flat_ratio": float(flat_ratio),
            "shortlist_m": int(m),
            "pred_samples_cheap": int(N_PRED_SAMPLES_CHEAP),
            "pred_samples_expensive": int(N_PRED_SAMPLES_EXPENSIVE),
        },
        "top_candidates_report": report_rows
    }

    # predicted stats for selected point (match on rounding)
    key = _rounded_key(next_x)
    jmatch = None
    for j in range(short_np.shape[0]):
        if _rounded_key(short_np[j]) == key:
            jmatch = j
            break

    if jmatch is not None:
        info["next_pred_mean"] = float(mean_raw[jmatch])
        info["next_pred_std"] = float(std_raw[jmatch])
        info["next_prob_improvement"] = float(pi_np[jmatch])
    else:
        info["next_pred_mean"] = float("nan")
        info["next_pred_std"] = float("nan")
        info["next_prob_improvement"] = float("nan")

    # interpretability: local sensitivity at x_next (posterior-mean gradient)
    try:
        sens = posterior_mean_sensitivity(model.__closure__[0].cell_contents, guide, next_x, device=device)
        # NOTE: model captures feature_extractor in closure; this is stable in this script
        info["local_sensitivity"] = sens
    except Exception:
        info["local_sensitivity"] = {"grad_dim0": float("nan"), "grad_dim1": float("nan"),
                                     "grad_l2": float("nan"), "grad_abs_sum": float("nan")}

    return next_x, info


# ------------------------ 9. Main ------------------------

def main():
    set_all_seeds(RANDOM_SEED)

    X_raw, y_raw = load_data()
    print(f"Loaded X_raw shape: {X_raw.shape}, y_raw shape: {y_raw.shape}")

    # --- Identify incumbent best (raw) for transparent baseline ---
    ib = int(np.argmax(y_raw))
    print("\n=== INCUMBENT BEST (RAW OBSERVED) ===")
    print(f"x_best_raw: [{X_raw[ib,0]:.6f} {X_raw[ib,1]:.6f}]")
    print(f"y_best_raw: {y_raw[ib]:.6g}")

    # Signed-log transform then standardize
    y_s = compute_scale_s(y_raw)
    y_trans = signed_log_transform(y_raw, y_s)

    scaler_y_trans = StandardScaler()
    y_scaled_np = scaler_y_trans.fit_transform(y_trans.reshape(-1, 1)).astype(np.float32).ravel()

    device = torch.device("cpu")
    X_t = torch.from_numpy(X_raw.astype(np.float32)).to(device)
    y_t = torch.from_numpy(y_scaled_np.astype(np.float32)).to(device)

    print("\n=== PRETRAIN FEATURE EXTRACTOR ===")
    feature_extractor = train_feature_extractor(X_t, y_t)
    feature_extractor.eval()
    print("Feature extractor trained.")

    print("\n=== HYPERPARAM TUNING (Random Search + 4-fold CV) ===")
    best_hp = tune_hyperparameters(feature_extractor, X_t, y_t)

    print("Best hyperparameters found:")
    print(f"  CV score (avg log-lik): {best_hp['score']:.6f}")
    print(f"  prior_scale:    {best_hp['prior_scale']:.6g}")
    print(f"  lr_vi:          {best_hp['lr_vi']:.6g}")
    print(f"  n_steps_vi:     {best_hp['n_steps_vi']}")
    print(f"  n_pred_samples: {best_hp['n_pred_samples']}")
    print(f"  xi (EI/PI):     {best_hp['xi']:.6g}")

    print("\n=== TRAIN FINAL BAYESIAN HEAD ON ALL DATA ===")
    bayes_cfg = BayesianHeadConfig(prior_scale=best_hp["prior_scale"])
    bayes_model = make_bayesian_model(feature_extractor, bayes_cfg)
    guide = train_bayesian_head(
        bayes_model, X_t, y_t,
        lr_vi=best_hp["lr_vi"],
        n_steps_vi=best_hp["n_steps_vi"]
    )
    print("Final Bayesian head training complete.")

    print("\n=== PROPOSE NEXT QUERY POINT (Week 10 transparency-aware) ===")
    next_x, info = propose_next_point_week10(
        bayes_model, guide,
        X_t, y_t,
        X_train_raw_np=X_raw,
        y_train_raw_np=y_raw,
        scaler_y_transformed=scaler_y_trans,
        y_scale_s=y_s,
        xi=best_hp["xi"],
    )

    # Print x_next to 6 decimals only
    x6 = np.round(np.array(next_x, dtype=np.float64), 6)
    x6 = np.clip(x6, 0.0, 1.0)

    print("\n=== CURRENT BEST (from observed data) ===")
    print(f"Best observed y (original scale): {info['best_y_raw']:.6g}")

    print("\n=== PROPOSED NEXT QUERY POINT ===")
    print(f"x_next (6 d.p., in [0,1]^2): [{x6[0]:.6f} {x6[1]:.6f}]")

    print("\n=== PREDICTIONS AT x_next (best-effort) ===")
    print(f"Predicted y mean (original scale): {info.get('next_pred_mean', float('nan')):.6g}")
    print(f"Predictive std (original scale):    {info.get('next_pred_std', float('nan')):.6g}")
    print(f"Prob. improvement over best:        {info.get('next_prob_improvement', float('nan')) * 100:.2f}%")

    print("\n=== COVERAGE AUDIT (data distribution check) ===")
    ca = info["meta"]["coverage_audit"]
    print(f"nn_min={ca['nn_min']:.4f}, nn_median={ca['nn_median']:.4f}, nn_mean={ca['nn_mean']:.4f}")
    print(f"spread_dim0={ca['spread_dim0']:.4f}, spread_dim1={ca['spread_dim1']:.4f}")
    print(f"local_mix_used={info['meta']['local_mix']:.2f}")

    print("\n=== TOP EI CONTENDERS (expensive stage, interpretable trace) ===")
    for r, row in enumerate(info["top_candidates_report"], start=1):
        x = row["x"]
        print(f"{r:>2}. x=[{x[0]:.6f} {x[1]:.6f}]  EI={row['EI']:.4e}  PI={row['PI']:.3f}  "
              f"mean={row['pred_mean_raw']:.4e}  std={row['pred_std_raw']:.4e}")

    print("\n=== LOCAL SENSITIVITY at x_next (posterior-mean gradient) ===")
    ls = info.get("local_sensitivity", {})
    print(f"dmu/dx0={ls.get('grad_dim0', float('nan')):.4e}, dmu/dx1={ls.get('grad_dim1', float('nan')):.4e}, "
          f"||grad||2={ls.get('grad_l2', float('nan')):.4e}")

    print("\n=== DECODING / ACQUISITION SETTINGS ===")
    ds = info["decode_settings"]
    print(f"temperature_used={ds['temperature_used']}, top_p={ds['top_p']}, top_k={ds['top_k']}")
    print(f"shortlist_m={ds['shortlist_m']}, pred_samples_cheap={ds['pred_samples_cheap']}, pred_samples_expensive={ds['pred_samples_expensive']}")
    print(f"flat_ratio={ds['flat_ratio']:.4f} (lower => flatter/less decisive)")

    print("\nDone.")


if __name__ == "__main__":
    main()


Loaded X_raw shape: (19, 2), y_raw shape: (19,)

=== INCUMBENT BEST (RAW OBSERVED) ===
x_best_raw: [0.546683 0.562315]
y_best_raw: 3.45699e-08

=== PRETRAIN FEATURE EXTRACTOR ===
Feature extractor trained.

=== HYPERPARAM TUNING (Random Search + 4-fold CV) ===
Best hyperparameters found:
  CV score (avg log-lik): -21.102266
  prior_scale:    1.15448
  lr_vi:          0.000905547
  n_steps_vi:     6000
  n_pred_samples: 256
  xi (EI/PI):     0.000127452

=== TRAIN FINAL BAYESIAN HEAD ON ALL DATA ===
Final Bayesian head training complete.

=== PROPOSE NEXT QUERY POINT (Week 10 transparency-aware) ===

=== CURRENT BEST (from observed data) ===
Best observed y (original scale): 3.457e-08

=== PROPOSED NEXT QUERY POINT ===
x_next (6 d.p., in [0,1]^2): [0.588448 0.633938]

=== PREDICTIONS AT x_next (best-effort) ===
Predicted y mean (original scale): 3.91623e-13
Predictive std (original scale):    354.023
Prob. improvement over best:        38.01%

=== COVERAGE AUDIT (data distribution check